# 12. Successor linkage and regional-reference evaluation

Generated: `2026-08-22T15:52:01`

This notebook is regenerated from the pipeline outputs. It is the reader-facing linkage and evaluation notebook.

## tl;dr

`M_B_text_ranking @ 0.70` is the frozen conservative primary event definition, not a claim of threshold optimality. Project history shows that the Grand Ouest regional reference informed the retained policy, so the comparison below is internal validation. All four algorithms are compared, including `M_D_fellegi_sunter`, scored from the fitted model.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'scripts').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
PROCESSED = PROJECT_ROOT / 'data/processed/boamp'
BENCHMARK = PROCESSED / 'regional_benchmark'

def load_json(path):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

dev = load_json(PROCESSED / 'linkage_evaluation_dev.json')
validation = load_json(PROCESSED / 'linkage_evaluation_validation.json')
modeling = load_json(BENCHMARK / 'modeling/modeling_summary.json')
manifest = load_json(BENCHMARK / 'regional_benchmark_manifest.json')


In [ ]:
def method_frame(summary):
    rows = []
    for method in summary['methods']:
        metrics = method['unweighted']
        weighted = method.get('design_weighted', {})
        rows.append({
            'method': method['method'],
            'threshold': method['threshold'],
            'accepted_links': metrics['accepted_links'],
            'precision': metrics['precision_at_1'],
            'recall': metrics['recall_at_1'],
            'fpr': metrics['false_positive_rate_on_negatives'],
            'coverage': metrics['coverage'],
            'precision_ci': metrics.get('precision_at_1_interval_95'),
            'recall_ci': metrics.get('recall_at_1_interval_95'),
            'weighted_precision': weighted.get('precision_at_1', {}).get('estimate'),
            'weighted_recall': weighted.get('recall_at_1', {}).get('estimate'),
            'weighted_fpr': weighted.get('false_positive_rate_on_verified_negatives', {}).get('estimate'),
        })
    return pd.DataFrame(rows)

dev_methods = method_frame(dev)
validation_methods = method_frame(validation)
validation_methods


## What Is Actually Being Scored

The decision here is **not** a yes/no call on a pre-formed pair, so an ordinary binary confusion matrix would hide the failure mode that matters most. For each anchor $i$ the reference names a *successor identity* and the method returns one, and the method can be wrong by naming the wrong candidate rather than by naming one at all.

Let $J_i$ be the candidate set that survived blocking for anchor $i$. Then

$$R_i \in J_i \cup \{\varnothing\}, \qquad \hat R_i \in J_i \cup \{\varnothing\},$$

where $R_i$ is the **reviewed successor** the reference identifies ($\varnothing$ if it found none) and $\hat R_i$ is the successor the linkage rule **accepts** ($\varnothing$ if it abstains). Three indicators follow, and they are what every metric below is built from:

$$P_i = \mathbf{1}\{R_i \neq \varnothing\}, \qquad A_i = \mathbf{1}\{\hat R_i \neq \varnothing\}, \qquad C_i = \mathbf{1}\{\hat R_i = R_i \neq \varnothing\}.$$

In words: $P_i$ says the reference found a successor, $A_i$ says the method committed to one, and $C_i$ says the one it committed to is exactly the reviewed one. A fourth indicator belongs to the stage *before* linkage:

$$E_i = \mathbf{1}\{R_i \in J_i\},$$

the reviewed successor survived candidate generation. An anchor with $E_i = 0$ is unrecoverable by any scorer, however good.

The reason this notation earns its place: an anchor can have $P_i = 1$ and $A_i = 1$ and still have $C_i = 0$, because the method accepted the wrong candidate. That single case is counted against precision *and* against recall, and it is invisible in the $TP/(TP+FP)$ shorthand.

## The Metrics As Conditional Probabilities

Each metric conditions on a different population, which is why they can move in opposite directions.

| Quantity | Form | Question it answers |
|---|---|---|
| Candidate reachability | $P(E=1 \mid P=1)$ | If a reviewed successor exists, did blocking keep it? |
| Precision | $P(C=1 \mid A=1)$ | If the method accepts, is the accepted candidate the reviewed one? |
| Recall | $P(C=1 \mid P=1)$ | If a reviewed successor exists, is it recovered exactly? |
| False-positive rate | $P(A=1 \mid P=0)$ | If no reviewed successor exists, did the method accept anyway? |
| Coverage | $P(A=1)$ | What share of anchors get a link rather than an abstention? |

**Precision and recall are the same event under reversed conditioning.** $P(C=1 \mid P=1)$ conditions on the reference; $P(C=1 \mid A=1)$ conditions on the algorithm. They are not interchangeable and need not be close, because the two conditioning sets are different populations. The false-positive rate conditions on a *third* population -- anchors the reference found nothing for -- so it is **not** $1 - \text{precision}$; the two share no denominator.

The two stages are deliberately given opposite objectives. Candidate generation maximises $P(E=1 \mid P=1)$, because a successor it discards is gone for good; the linkage rule then maximises $P(C=1 \mid A=1)$, because a false link fabricates both a survival event and an event time. High recall first, high precision later. This also fixes the ceiling: since a correct acceptance requires the successor to be in $J_i$ at all,

$$P(C=1 \mid P=1) \le P(E=1 \mid P=1).$$

## Reference State

In [ ]:
pd.DataFrame([
    {'item': 'reviewed anchors', 'value': manifest['reviewed_anchors']},
    {'item': 'resolved to one episode', 'value': manifest['remap']['resolved_to_current_episodes']},
    {'item': 'pilot usable anchors', 'value': manifest['splits']['dev']['usable_anchors']},
    {'item': 'pilot positive anchors', 'value': manifest['splits']['dev']['positive_anchors']},
    {'item': 'locked usable anchors', 'value': manifest['splits']['validation']['usable_anchors']},
    {'item': 'locked positive anchors', 'value': manifest['splits']['validation']['positive_anchors']},
    {'item': 'reviewed successors in the reference', 'value': manifest['candidate_reachability']['positive_anchors']},
    {'item': 'of those, reachable after blocking', 'value': manifest['candidate_reachability']['positive_anchors_with_reviewed_successor_in_pool']},
    {'item': 'P(E=1 | P=1), the recall ceiling', 'value': manifest['candidate_reachability']['candidate_generation_recall_ceiling']},
])

Candidate generation retained the reviewed successor in `21` of the `23` reviewed cases, so $\hat P(E=1 \mid P=1) = 21/23 = 0.913$. This is candidate-generation reachability measured on this reference sample -- *pairs completeness* in record-linkage terms -- and **not** population recall. Both unreachable cases are attributed to a named blocking condition in `CANDIDATE_GENERATION_AUDIT.md`; neither is an implementation defect.

## Internal-Validation Comparison On The Recorded Locked Stratum

In [ ]:
display(validation_methods[['method', 'threshold', 'accepted_links', 'precision', 'precision_ci', 'recall', 'recall_ci', 'fpr', 'coverage']])

ax = validation_methods.set_index('method')[['precision', 'recall', 'fpr']].plot(
    kind='bar', figsize=(9, 4.5), width=0.72
)
ax.set_title('Locked split: the same three conditional probabilities per method')
ax.set_ylabel('probability')
ax.set_ylim(0, 1)
ax.set_xlabel('')
ax.legend(['precision  P(C=1 | A=1)', 'recall  P(C=1 | P=1)',
           'FPR  P(A=1 | P=0)'], frameon=False)
ax.tick_params(axis='x', rotation=28)
ax.grid(axis='y', alpha=0.25)
plt.tight_layout()

In [ ]:
# The frozen rule read cell by cell, so each rate above can be traced to
# the anchors that produced it.
m_b = next(m['unweighted'] for m in validation['methods']
           if m['method'] == 'M_B_text_ranking')
display(pd.DataFrame([
    {'cell': 'C=1 (accepted the reviewed successor)', 'anchors': m_b['true_positive']},
    {'cell': 'A=1, P=1, C=0 (accepted the wrong candidate)', 'anchors': m_b['false_positive_wrong_successor']},
    {'cell': 'A=0, P=1 (abstained on a positive anchor)', 'anchors': m_b['false_negative_abstained']},
    {'cell': 'A=1, P=0 (accepted where the reference has none)', 'anchors': m_b['false_positive_on_no_successor_anchor']},
    {'cell': 'A=0, P=0 (abstained on a negative anchor)', 'anchors': m_b['true_negative_abstained']},
]).set_index('cell'))

print(f"P(C=1 | A=1) = {m_b['true_positive']}/{m_b['accepted_links']}"
      f" = {m_b['precision_at_1']:.3f}   precision")
print(f"P(C=1 | P=1) = {m_b['true_positive']}/{m_b['positive_anchors']}"
      f" = {m_b['recall_at_1']:.3f}   recall")
print(f"P(A=1 | P=0) = {m_b['false_positive_on_no_successor_anchor']}/{m_b['negative_anchors']}"
      f" = {m_b['false_positive_rate_on_negatives']:.3f}   false-positive rate")
print(f"P(A=1)       = {m_b['accepted_links']}/{m_b['anchors_evaluated']}"
      f" = {m_b['coverage']:.3f}   coverage")


### Reading those four numbers

**Precision, $\hat P(C=1 \mid A=1) = 7/8 = 0.875$.** Among the `8` links `M_B` accepted on the locked reference, `7` matched the reviewed successor. This is a reference-sample estimate on `8` accepted links with a wide interval (95% CI `0.529`-`0.978`): one changed decision moves it materially. It is not population accuracy and not independent specialist validation.

**Recall, $\hat P(C=1 \mid P=1) = 7/18 = 0.389$.** Of the `18` anchors the reference says have a successor, the rule recovered `7`. The `11` misses are of two kinds and count identically here: `10` abstentions and `1` wrong acceptance. Only the second fabricates a link. The ceiling is not 1 but `0.913`, set by blocking.

**False-positive rate, $\hat P(A=1 \mid P=0) = 0/54 = 0.000$.** On the `54` anchors the reference found nothing for, the rule accepted nothing either. This is a diagnostic on a small set of corpus-relative negatives, **not** evidence that the population false-positive rate is literally zero.

**Coverage, $\hat P(A=1) = 8/72 = 0.111$.** Roughly one anchor in nine receives a link. Low coverage is not a defect to be fixed here: it is the direct consequence of prioritising $P(C=1 \mid A=1)$, and in a survival dataset an abstention becomes honest right-censoring while a false link becomes a fabricated event at a fabricated time.

## Interpretation

`M_C_weighted_gated` recovers more reviewed successors, but its false-positive rate is materially higher. For survival analysis, a false link is more damaging than an abstention because it fabricates both an event and an event time. Thresholds other than `0.70` are carried as sensitivity arms rather than selected from these rows: replacing the frozen policy requires fresh independent evidence. On a reference this small the intervals overlap heavily, so read them before separating any two methods. The use of precision-recall evidence for this rare-positive decision follows [Davis and Goadrich (2006)](https://doi.org/10.1145/1143844.1143874) and [Saito and Rehmsmeier (2015)](https://doi.org/10.1371/journal.pone.0118432). Those papers support the diagnostic choice, not this project's numerical results.

## Modeling-Ready Tables

In [ ]:
pd.DataFrame(modeling['outputs']).T[[
    'rows', 'anchors', 'primary_positive_pairs', 'positive_anchors'
]]

In [ ]:
feature_columns = pd.Series(modeling['feature_columns'], name='feature')
display(feature_columns.to_frame())
assert 'fs_match_probability' in set(modeling['feature_columns'])


## Caveat

The labels were generated by a single LLM research pass over real BOAMP notices, their official URLs, and wider public sources, dated 2026-08-11, then spot-checked on a subset by the project owner. They are independent of every method scored here, but they were not verified anchor-by-anchor and are not an independent specialist panel. Negatives are corpus-relative: roughly 25 candidates per anchor were considered, so the false-positive rate is conservative by construction rather than a population-wide rate. These are reference-sample estimates, not validated legal-renewal accuracy.